## A special cost function

We implement a piecewise defined cost function.
We use quartic polynomials in our cost.
Let $[-a, a] \subseteq \mathbb{R}$ be a symmetric interval, let $x_0 = 0 < x_1 < \ldots < x_n = a$ denote a partition of $[0, a]$, and let $y_0 < y_1 < \ldots < y_n$ denote corresponding outputs.
Let $p: [-a, a] \to \mathbb{R}$ be a polynomial such that the following hold.

1. $p(x)$ is a quartic polynomial on each interval
2. $p(x) = p(-x)$ is symmetric
3. $p(x_k) = y_k$ for all $0 \leq k \leq n$
4. $p(x)$ is $\mathcal{C}^2([-u, u])$
5. $p'(0) = 0$ and $p''(x) \geq 0$
6. the quantity $\int_{-a}^a |p''(x)|^2 \operatorname{d} x$ is minimized over all polynomials satisfying properties 5-6.

Of course, such a polynomial should be computed via a minimization routine in SciPy.

TL;DR: Apparently, "Of course, ..." was not obvious.
SciPy handled the approximation quite poorly, and there were many subtleties about when a solution even exists.
A naive solution is adopted below, after a lot of effort is spent 
The naive solution is refined into a JAX implementation.

In [ ]:
from __future__ import annotations

In [ ]:
%matplotlib ipympl

import dataclasses
import functools
import itertools
import typing as tp
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import scipy.optimize as sci_opt
import IPython.display as ipd

jax.config.update("jax_enable_x64", True)

## setup

We choose to perform many of the symbolic computations in SymPy.
E.g., to compute the integral of a polynomial, first integrate in SymPy, then call `lambdify` with JAX from SymPy, and then use that function in another function.

In [ ]:
# vars
coeff_vars = sp.symbols("a_0 a_1 a_2 a_3 a_4")
x_var = sp.Symbol("x")

# poly
# note the numpy.polyval assumes the coefficients are in reverse order
# cf. https://docs.jax.dev/en/latest/_autosummary/jax.numpy.polyval.html
n = len(coeff_vars)
quartic_sym_expr = sum(
    [coeff_vars[n - 1 - i] * x_var**i for i in range(n)],
)
quartic_sym = sp.Poly(quartic_sym_expr, x_var)
quartic_sym

In [ ]:
# helper(s)

def is_tracer(x: tp.Any) -> bool:
    """Check if x is a JAX tracer."""
    return isinstance(x, jax.core.Tracer)  # type: ignore

## variable dataclass

Pretty standard implementation here.
Note that this could be made more efficient.
The knots can be made static (because we don't care about moving them during the optimization process).
Also, because the knots are static, the coefficients should be reparameterized to a unit interval (shift and scale the input variable, so that the coefficients are better behaved).
Cf. https://numpy.org/doc/stable/reference/routines.polynomials.html.

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class PiecewisePoly:
    """Quartic piecewise polynomial."""
    knots: jax.Array  # can make static...
    coeffs: jax.Array

    @classmethod
    def from_flat(cls, knots: jax.Array, flat: jax.Array) -> "PiecewisePoly":
        coeffs = jnp.ravel(flat)
        assert coeffs.size % 5 == 0  # quartic polynomial
        coeffs = jnp.reshape(coeffs, (-1, 5))
        assert coeffs.shape[0] == knots.size - 1  # segments

        # the following do not work with tracers...
        # assert knots[0] == 0  # first knot is zero
        # assert jnp.all(knots >= 0)  # non-negative knots
        # assert jnp.all(jnp.diff(knots) > 0)  # sorted knots

        return cls(knots=knots, coeffs=coeffs)

    def flatten(self) -> jax.Array:
        """Convert a piecewise polynomial dataclass to a flat array."""
        return jnp.ravel(self.coeffs)

    def eval_at_index(self, x: jax.Array, nu: int, index: jax.Array) -> jax.Array:
        # differentiate polynomial (symbolically)
        coeffs = self.coeffs.at[index, :].get()
        coeffs_sym = quartic_sym.diff((0, nu)).coeffs()
        coeffs_lambda = sp.lambdify(
            coeff_vars,
            sp.Matrix(coeffs_sym),
            modules="jax",
        )
        coeffs_nu = jnp.ravel(coeffs_lambda(*coeffs))
        return jnp.polyval(coeffs_nu, x)

    @functools.partial(jax.jit, static_argnames=("nu",))
    def __call__(self, x: jax.Array, nu: int = 0) -> jax.Array:
        """Evaluate the piecewise polynomial at x.

        Hindsight shows that we could have just used jax.grad instead of
        including the nu argument.
        """
        assert nu >= 0
        x = jnp.ravel(x).reshape()  # enforce scalar
        x = jnp.abs(x)  # enforce poly symmetry
        index = jnp.searchsorted(self.knots, x) - 1
        index = jnp.clip(index, 0, self.knots.size - 2)  # clip to valid range
        return self.eval_at_index(x, nu, index)


## loss

In [ ]:
@jax.jit
def poly_curvature_loss(poly: PiecewisePoly) -> jax.Array:
    integrand_loss = quartic_sym.diff((0, 2))**2
    integral_loss = integrand_loss.integrate((0, 1))
    integral_coeffs = integral_loss.all_coeffs()  # need all, including zeros

    def _coeffs_eval(_coeffs: jax.Array) -> jax.Array:
        _sp_fun = sp.lambdify(
            coeff_vars,
            sp.Matrix(integral_coeffs),
            modules="jax",
        )
        return jnp.ravel(_sp_fun(*_coeffs))

    coeffs_eval = jax.vmap(_coeffs_eval, in_axes=0)(poly.coeffs)
    int_a_fun = jax.vmap(
        lambda _coeffs, _knot: jnp.polyval(_coeffs, _knot)
    )
    int_a = int_a_fun(coeffs_eval, poly.knots.at[:-1].get())
    int_b_fun = jax.vmap(
        lambda _coeffs, _knot: jnp.polyval(_coeffs, _knot)
    )
    int_b = int_b_fun(coeffs_eval, poly.knots.at[1:].get())
    return jnp.sum(int_b - int_a)


## constraints

Apparently, all the constraints are linear.
Technically, the constraint $p''(x) \geq 0$ is not linear, but we approximate it with a linear constraint.
See discussion below.

In [ ]:
# linear
@jax.jit
def point_constraint(y: jax.Array, poly: PiecewisePoly) -> jax.Array:
    """Knot matching."""
    y = jnp.ravel(y)
    assert y.size == poly.knots.size
    y_poly = jax.vmap(poly)(poly.knots)
    return y_poly - y


In [ ]:
# linear
@jax.jit
def c2_constraint(poly: PiecewisePoly) -> jax.Array:
    """Enforce smoothness (upto 2nd derivatives) at knots."""
    def _knot_derivatives(_index: jax.Array) -> jax.Array:
        _knot = poly.knots.at[_index].get()
        _left = jnp.array([
            poly.eval_at_index(_knot, nu=0, index=_index - 1),
            poly.eval_at_index(_knot, nu=1, index=_index - 1),
            poly.eval_at_index(_knot, nu=2, index=_index - 1),
        ])
        _right = jnp.array([
            poly.eval_at_index(_knot, nu=0, index=_index),
            poly.eval_at_index(_knot, nu=1, index=_index),
            poly.eval_at_index(_knot, nu=2, index=_index),
        ])
        return _left - _right
    
    res = jax.vmap(_knot_derivatives)(jnp.arange(1, poly.knots.size - 1))
    return jnp.ravel(res)


In [ ]:
# linear
@jax.jit
def origin_constraint(poly: PiecewisePoly) -> jax.Array:
    """Enforce origin constraint."""
    return jnp.array([poly(jnp.array(0.0), nu=0),
                       poly(jnp.array(0.0), nu=1),
                       poly(jnp.array(0.0), nu=2)])

TL;DR: read the next markdown block.

Let $p(x)$ be one of our piecewise quartics.
The constraint $p''(x) \geq 0$ on $[-u, u]$ is somewhat subtle to implement.
Because $p(x)$ is guaranteed to be quartic, then $p''(x)$ is quadratic.
Fix $x_0 \in [-u, u]$, and let $x_0$ lie in the partition $[t_k, t_{k + 1}] \ni x_0$.
Let $p_{x_0}(x)$ be the quadratic polynomial such that $p_{x_0}(x) = p(x)$ on $[t_k, t_{k + 1}]$.
We do _not_ need $p_{x_0}''(x) \geq 0$ on all of $\mathbb{R}$, but rather only on $[t_k, t_{k + 1}]$.
This implies cases.
Let $m_{x_0} \in \mathbb{R}$ denote local extrema of the quadratic, defined by the quadratic equation, in the non-degenerate case.
I.e., let $m_{x_0}$ denote _the_ root of $p_{x_0}'''$.
Let $p_{x_0}(x) = a_0 \, x^4 + a_1 \, x^3 + a_2 \, x^2 + a_3 \, x + a_4$.
Explicitly, there are two cases.

1. If $m_{x_0}$ exists and if $p_{x_0}^{(4)} \equiv \mathrm{const} \geq 0 (= p^{(4)}(x_0))$, then check that $p''_{x_0}(m_{x_0}) \geq 0$.
I.e., for this case, we need $a_0 > 0$.
In practice, we check $a_0 > 2^{-10}$.

2. Otherwise, check that $p_{x_0}''(t_k) \geq 0$ and $p''(t_{k + 1}) \geq 0$.


However, there is another subtlety.
As previously listed, our constraint function will return arrays of different sizes: of size 1 in case 1 and of size 2 in case 2.
Note that case 2 will always hold if case 1 is valid.

We can artificially always return a 3 vector, but it is subtle to guarantee global smoothness (in cases where $a_0 = 0$ or when $m_{x_0} \not\in (t_k, t_{k + 1})$).
Instead, we evaluate at $3$ uniformly spaced points on the interior ($5$ points, including boundary).
(This magic number can be used as a hyperparameter, if necessary.)
We could make this more robust by shifting the points depending on the location of $m_{x_0}$ (globally) and add a constraint to force $a_0 > 0$.
But the former is easier to implement.

In [ ]:
# linear
@jax.jit
def curvature_constraint(poly: PiecewisePoly) -> jax.Array:
    """Enforce (approximately) curvature constraint."""
    @jax.vmap
    def _samples(_index: jax.Array) -> jax.Array:
        _left = poly.knots.at[_index].get()
        _right = poly.knots.at[_index + 1].get()
        return jnp.linspace(_left, _right, 2**3)  # magic number (samples)

    @functools.partial(jax.vmap, in_axes=(0, 0))
    def _derivatives(_x: jax.Array, index: jax.Array) -> jax.Array:
        return jax.vmap(poly.eval_at_index, in_axes=(0, None, None))(
            _x, 2, index
        )

    indices = jnp.arange(poly.knots.size - 1)
    samples = _samples(indices)
    derivatives = _derivatives(samples, indices)
    return jnp.ravel(derivatives)


## wrappers

In [ ]:
def jax_poly_curvature_loss(
    knots: jax.Array, coeffs: jax.Array
) -> jax.Array:
    """Loss wrapper."""
    coeffs = jnp.ravel(coeffs)
    poly = PiecewisePoly.from_flat(
        knots=knots,
        flat=coeffs,
    )
    return poly_curvature_loss(poly)


def np_poly_curvature_loss(knots: np.ndarray, coeffs: np.ndarray) -> float:
    """Loss wrapper."""
    loss_fun = jax.jit(functools.partial(
        jax_poly_curvature_loss, jnp.array(knots, dtype=float)
    ))
    return float(loss_fun(jnp.array(coeffs, dtype=float)))


def np_poly_curvature_loss_jac(
    knots: np.ndarray, coeffs: np.ndarray
) -> np.ndarray:
    """Loss jacobian wrapper."""
    loss_fun = functools.partial(
        jax_poly_curvature_loss, jnp.array(knots, dtype=float)
    )
    grad_fun = jax.jit(jax.grad(loss_fun))
    return np.array(grad_fun(jnp.array(coeffs, dtype=float)))


# note the 'p' suffix
def np_poly_curvature_loss_hessp(
    knots: np.ndarray, coeffs: np.ndarray, v: np.ndarray
) -> np.ndarray:
    """Loss hessp wrapper."""
    loss_fun = functools.partial(
        jax_poly_curvature_loss, jnp.array(knots, dtype=float)
    )
    primals = (jnp.array(coeffs, dtype=float),)
    tangents = (jnp.array(v, dtype=float),)
    return np.array(jax.jvp(jax.grad(loss_fun), primals, tangents)[1])


In [ ]:
def np_constraint_eq(
    y: np.ndarray, knots: np.ndarray,
) -> sci_opt.LinearConstraint:
    knots = np.ravel(knots)
    y = np.ravel(y)
    assert knots.size == y.size

    def _constraint_fun(_coeffs: jax.Array) -> jax.Array:
        poly = PiecewisePoly.from_flat(
            knots=jnp.array(knots, dtype=float), flat=_coeffs
        )
        return jnp.concatenate([
            point_constraint(y=jnp.array(y), poly=poly),
            c2_constraint(poly),
            origin_constraint(poly),
            # curvature_constraint(poly),  # inequality constraint
        ])
    
    num_coeffs = (knots.size - 1) * (4 + 1)
    mat = jax.jacobian(_constraint_fun)(jnp.zeros(num_coeffs))
    bounds = np.concatenate([
        y, np.zeros(mat.shape[0] - y.size)
    ])
    return sci_opt.LinearConstraint(
        A=np.array(mat),
        lb=bounds,  # type: ignore
        ub=bounds,  # type: ignore
        keep_feasible=False,
    )


def np_constraint_ineq(knots: np.ndarray) -> sci_opt.LinearConstraint:

    def _constraint_fun(_coeffs: jax.Array) -> jax.Array:
        poly = PiecewisePoly.from_flat(
            knots=jnp.array(knots, dtype=float), flat=_coeffs
        )
        return curvature_constraint(poly)
    
    num_coeffs = (knots.size - 1) * (4 + 1)
    mat = jax.jacobian(_constraint_fun)(jnp.zeros(num_coeffs))
    return sci_opt.LinearConstraint(
        A=np.array(mat),
        lb=np.zeros(mat.shape[0]),  # type: ignore
        ub=np.inf * np.ones(mat.shape[0]),  # type: ignore
        keep_feasible=False,
    )


## testing

In [ ]:
# y = np.array([0.0, 1.0, 4.0, 8.0])
# knots = np.array([0.0, 1.0, 2.0, 3.0])

y = np.array([0.0, 1.0, 8.0])
knots = np.array([0.0, 1.0, 2.0])

In [ ]:
res_guess = sci_opt.minimize(
    fun=functools.partial(np_poly_curvature_loss, knots),
    jac=functools.partial(np_poly_curvature_loss_jac, knots),
    hessp=functools.partial(np_poly_curvature_loss_hessp, knots),
    x0=np.zeros(5 * (knots.size - 1)),
    method="trust-constr",
    constraints=[
        np_constraint_eq(y, knots),
        # np_constraint_ineq(knots),  # no concavity constraints
    ],
    options={
        # "verbose": 3,
        "gtol": 1e-6,
        "xtol": 1e-6,
        "maxiter": 100,
        "initial_tr_radius": 1.0,
    },
)
res_guess

In [ ]:
res = sci_opt.minimize(
    fun=functools.partial(np_poly_curvature_loss, knots),
    jac=functools.partial(np_poly_curvature_loss_jac, knots),
    hessp=functools.partial(np_poly_curvature_loss_hessp, knots),
    x0=res_guess.x,
    method="trust-constr",
    constraints=[
        np_constraint_eq(y, knots),
        np_constraint_ineq(knots),  # no concavity constraints
    ],
    options={
        # "verbose": 3,
        "gtol": 1e-8,
        "xtol": 1e-8,
        "maxiter": 100,
        "initial_tr_radius": 1.0,
    },
)
res

In [ ]:
plt.close("all")

In [ ]:
poly_guess = PiecewisePoly.from_flat(
    knots=jnp.array(knots, dtype=float),
    flat=jnp.array(res_guess.x, dtype=float),
)
poly = PiecewisePoly.from_flat(
    knots=jnp.array(knots, dtype=float),
    flat=jnp.array(res.x, dtype=float),
)

# x = np.linspace(-10.0, 10.0, 2**18)
# x = np.linspace(-3.0, 3.0, 2**18)
x = np.linspace(-2.0, 2.0, 2**18)
# x = np.linspace(-1.0, 1.0, 2**18)
# x = np.linspace(-0.5, 0.5, 2**18)
# x = np.linspace(-0.1, 0.1, 2**18)
nu = 0

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, jax.vmap(functools.partial(poly_guess, nu=nu))(x), label="poly_guess")
ax.plot(x, jax.vmap(functools.partial(poly, nu=nu))(x), label="poly")
ax.plot(x, x**4, label="quartic")
ax.plot(x, x**2, label="quadratic")
ax.legend()
plt.show()


## a naive approach

We provide a handcrafted solution, where SymPy helped with some computations.
This is the analytic solution that the block-box nonlinear solver seemed to fail to find.

In [ ]:
# def worst_a1(a4, a3, a2, y, t0, t1):
#     return -2 * a2


# def worst_a0(a4, a3, a2, a1, y, t0, t1):
#     return a2 / (t1 - t0)


# def analytic_a1(a4, a3, a2, y, t0, t1):
#     a1 = 11 * a2 * (t1 - t0)**2 + 27 * a3 * (t1 - t0) - 27 * y
#     a1 /= -2 * (t1 - t0)**3
#     return a1


# def analytic_a0(a4, a3, a2, a1, y, t0, t1):
#     a0 = a1 * (t0 - t1)**3 - 3 * a2 * (t0 -  t1)**2 + 6 * a3 * (t0 - t1) -6 * a4 + 6 * y
#     a0 /= 0.5 * (t0 - t1)**4
#     return a0

In [ ]:
def worst_a1(a4, a3, a2, y, t0, t1):
    return 0.0


def worst_a0(a4, a3, a2, a1, y, t0, t1):
    return 0.0


def analytic_a1(a4, a3, a2, y, t0, t1):
    return 0.0


def analytic_a0(a4, a3, a2, a1, y, t0, t1):
    a0 = y - a4 - a3 * (t1 - t0) - 1.0 / 2.0 * a2 * (t1 - t0)**2
    # - 1.0 / 6.0 * a1 * (t1 - t0)**3
    a0 /= (t1 - t0)**4 / 12.0
    return a0

In [ ]:
def polyval_special(coeffs: list[float], x: float, t0: float) -> float:
    assert len(coeffs) == 5
    coeffs = [
        1 / 12 * coeffs[0],
        1 / 6 * coeffs[1],
        1 / 2 * coeffs[2],
        coeffs[3],
        coeffs[4],
    ]
    return float(np.polyval(coeffs, x - t0))


def polyval_specialp(coeffs: list[float], x: float, t0: float) -> float:
    assert len(coeffs) == 5
    coeffs = [
        1 / 3 * coeffs[0],
        1 / 2 * coeffs[1],
        coeffs[2],
        coeffs[3],
    ]
    return float(np.polyval(coeffs, x - t0))


def polyval_specialpp(coeffs: list[float], x: float, t0: float) -> float:
    assert len(coeffs) == 5
    coeffs = [
        coeffs[0],
        coeffs[1],
        coeffs[2],
    ]
    return float(np.polyval(coeffs, x - t0))


def quartic_cost(
    knots: np.ndarray,
    y: np.ndarray,
) -> np.ndarray:
    """Quartic cost function."""
    # coeffs[k] = [a_0, a_1, a_2, a_3, a_4]
    # p_k(x) = (1/12) a_0 (x - t_k)^4 + (1/6) a_1 (x - t_k)^3 + (1/2) a_2 (x - t_k)^2 + a_3 (x - t_k) + a_4
    a4 = 0.0
    a3 = 0.0
    a2 = 0.0
    a1 = analytic_a1(a4, a3, a2, y[1], knots[0], knots[1])
    a0 = analytic_a0(a4, a3, a2, a1, y[1], knots[0], knots[1])
    coeffs_0 = [a0, a1, a2, a3, a4]
    coeffs = [coeffs_0]

    for k in range(1, knots.size - 1):
        a4 = polyval_special(coeffs[k - 1], knots[k], knots[k - 1])
        a3 = polyval_specialp(coeffs[k - 1], knots[k], knots[k - 1])
        a2 = polyval_specialpp(coeffs[k - 1], knots[k], knots[k - 1])

        a1_bad = worst_a1(a4, a3, a2, y[k + 1], knots[k], knots[k + 1])
        a0_bad = worst_a0(a4, a3, a2, a1_bad, y[k + 1], knots[k], knots[k + 1])
        coeffs_k_bad = [a0_bad, a1_bad, a2, a3, a4]
        ell_k = polyval_special(coeffs_k_bad, knots[k + 1], knots[k])
        assert y[k + 1] >= ell_k, f"y[k + 1] = {y[k + 1]}, ell_k = {ell_k}"

        a1 = analytic_a1(a4, a3, a2, y[k + 1], knots[k], knots[k + 1])
        a0 = analytic_a0(a4, a3, a2, a1, y[k + 1], knots[k], knots[k + 1])

        coeffs_k = [a0, a1, a2, a3, a4]
        assert a0 >= 0.0, f"a0 = {a0}"
        coeffs.append(coeffs_k)
    coeffs = np.array(coeffs)
    return coeffs


In [ ]:
def standardize_coeffs(coeffs: np.ndarray, knots: np.ndarray) -> np.ndarray:
    def _standardize_coeffs(_coeffs: np.ndarray, _t0: float) -> np.ndarray:
        """Single selection of coefficients."""
        assert _coeffs.size == 5
        _a0, _a1, _a2, _a3, _a4 = _coeffs
        return np.array(
            [
                _a0 / 12,
                -_a0 * _t0 / 3 + _a1 / 6,
                _a0 * _t0**2 / 2 - _a1 * _t0 / 2 + _a2 / 2,
                -_a0 * _t0**3 / 3 + _a1 * _t0**2 / 2 - _a2 * _t0 + _a3,
                (
                    _a0 * _t0**4 / 12
                    - _a1 * _t0**3 / 6
                    + _a2 * _t0**2 / 2
                    - _a3 * _t0
                    + _a4
                ),
            ]
        )

    assert coeffs.shape[1] == 5
    assert coeffs.shape[0] == knots.size - 1
    standard_coeffs = []
    for c, k in zip(coeffs, knots[:-1]):
        standard_coeffs.append(_standardize_coeffs(c, k))
    return np.array(standard_coeffs)


In [ ]:
y = np.array([0.0, 1.0, 2**4, 2**8])
knots = np.array([0.0, 1.0, 2.0, 3.0])
coeffs = quartic_cost(knots, y)
coeffs = standardize_coeffs(coeffs, knots)
coeffs

In [ ]:
poly = PiecewisePoly(
    knots=jnp.array(knots, dtype=float),
    coeffs=jnp.array(coeffs, dtype=float),
)
x = np.linspace(-10.0, 10.0, 2**18)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, jax.vmap(functools.partial(poly, nu=0))(x), label="poly")
# ax.plot(x, jax.vmap(functools.partial(poly, nu=1))(x), label="poly")
# ax.plot(x, jax.vmap(functools.partial(poly, nu=2))(x), label="poly")
plt.show()

## a naive approach, in jax

We implement the naive approach in JAX.

In [ ]:
@jax.tree_util.register_dataclass
@dataclasses.dataclass
class QuarticCost:
    coeffs: jax.Array
    knots: jax.Array
    low: jax.Array
    high: jax.Array

    @classmethod
    def from_bounds(
        cls,
        margins: list[float],
        sizes: list[float],
        low: float,
        high: float,
    ) -> "QuarticCost":
        assert len(margins) + 1 == len(sizes)
        assert sorted(sizes) == sizes
        assert sum(margins) <= 1.0
        assert low < high

        unity_knots = [1.0]
        for m in margins:
            unity_knots.append(unity_knots[-1] - m)
        unity_knots.append(0.0)
        unity_knots.reverse()
        sizes = [0.0] + sizes
        coeffs = quartic_cost(
            knots=np.array(unity_knots),
            y=np.array(sizes),
        )
        # an oddity of the return type of quartic cost means that we need to
        #  scale the coefficients (derived from integrating twice)
        scale = np.array([1.0 / 12.0, 1.0 / 6.0, 1.0 / 2.0, 1.0, 1.0])
        coeffs *= np.tile(A=scale, reps=(coeffs.shape[0], 1))
        return cls(
            coeffs=jnp.array(coeffs, dtype=float),
            knots=jnp.array(unity_knots, dtype=float),
            low=jnp.array(low, dtype=float),
            high=jnp.array(high, dtype=float),
        )

    @jax.jit
    def __call__(self, x: jax.Array) -> jax.Array:
        assert x.size == 1
        x = jnp.ravel(x).reshape()

        # scale x to [0, 1], where the coefficients are defined
        mid = (self.high + self.low) / 2.0
        width = (self.high - self.low) / 2.0
        x = jnp.abs(x - mid) / width

        index = jnp.searchsorted(self.knots, x) - 1
        index = jnp.clip(index, 0, self.knots.size - 2)
        coeffs = self.coeffs.at[index, :].get()
        knot = self.knots.at[index].get()
        return jnp.polyval(coeffs, x - knot)


In [ ]:
qc = QuarticCost.from_bounds(
    margins=[0.2, 0.1],
    sizes=[1.0, 2**3, 2**8],
    low=-1.0,
    high=3.0,
)

x = jnp.linspace(-1.0, 3.0, 2**18)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, jax.vmap(qc)(x), label="poly")
# ax.plot(x, jax.vmap(jax.grad((qc)))(x), label="poly")
# ax.plot(x, jax.vmap(jax.grad(jax.grad((qc))))(x), label="poly")
plt.show()